# Notebook 01: YOLO11s-P2 Baseline

**Model**: YOLO11s-P2 (no modifications)

**Purpose**: Establish baseline performance for road damage detection with P2 detection scale.

In [1]:
import sys
import torch
import platform

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DEVICE = 0
    print(f"Current device: cuda:0")
else:
    print("CUDA NOT AVAILABLE - Training will proceed on CPU")
    print(f"Reason: PyTorch build = {torch.__version__} (CPU-only build)")
    DEVICE = "cpu"
    print(f"Current device: cpu")
print(f"OS: {platform.system()} {platform.release()}")
print("=" * 60)


ENVIRONMENT CHECK
Python version: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
PyTorch version: 2.11.0+cu128
Ultralytics version: 8.4.92


CUDA available: True
CUDA version: 12.8
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU
GPU memory: 6.00 GB
Current device: cuda:0
OS: Windows 11


In [2]:
# Shared training configuration - MUST be identical for all 5 models
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_CONFIG = {
    "data": r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\data.yaml",
    "imgsz": 640,
    "epochs": 100,
    "patience": 20,
    "batch": 8,
    "device": "cuda",
    "workers": 0,  # Windows safety
    "seed": SEED,
    "deterministic": True,
    "amp": True,  # AMP only with CUDA
    "pretrained": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "verbose": True,
}

RESULTS_DIR = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Training configuration loaded:")
for k, v in TRAIN_CONFIG.items():
    if k != "data":
        print(f"  {k}: {v}")


Training configuration loaded:
  imgsz: 640
  epochs: 100
  patience: 20
  batch: 8
  device: cuda
  workers: 0
  seed: 42
  deterministic: True
  amp: True
  pretrained: True
  optimizer: auto
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3.0
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  box: 7.5
  cls: 0.5
  dfl: 1.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.0
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 1.0
  mixup: 0.0
  copy_paste: 0.0
  verbose: True


In [3]:
def benchmark_model(model_path, device, imgsz=640, warmup=20, runs=100):
    """Controlled latency benchmark for a YOLO model."""
    import time
    from ultralytics import YOLO
    
    model = YOLO(model_path)
    
    # Create dummy input
    dummy = torch.randn(1, 3, imgsz, imgsz)
    if device != "cpu":
        dummy = dummy.to(f"cuda:{device}")
    
    # Warmup
    print(f"Warming up ({warmup} iterations)...")
    for _ in range(warmup):
        _ = model.predict(source=dummy, verbose=False, device=device)
    
    # Timed runs
    print(f"Benchmarking ({runs} iterations)...")
    latencies = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model.predict(source=dummy, verbose=False, device=device)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # ms
    
    mean_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    fps = 1000.0 / mean_lat
    
    print(f"Latency: {mean_lat:.2f} +/- {std_lat:.2f} ms")
    print(f"FPS: {fps:.1f}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return mean_lat, std_lat, fps


In [4]:
def save_experiment_results(model_name, results, model_path, benchmark_results, results_dir):
    """Save experiment results to JSON for comparison notebook."""
    import os, json
    
    mean_lat, std_lat, fps = benchmark_results
    
    # Get model file size
    model_size_bytes = os.path.getsize(model_path)
    model_size_mb = model_size_bytes / (1024 * 1024)
    
    # Get model info
    from ultralytics import YOLO
    model = YOLO(model_path)
    info = model.info()
    if isinstance(info, tuple) and len(info) >= 4:
        n_layers, n_params, n_grads, gflops = info[:4]
    else:
        n_layers = len(list(model.model.modules()))
        n_params = sum(p.numel() for p in model.model.parameters())
        n_grads = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        gflops = 0.0
    
    # Extract metrics from results
    metrics = {}
    if hasattr(results, 'results_dict'):
        rd = results.results_dict
        metrics["precision"] = rd.get("metrics/precision(B)", 0)
        metrics["recall"] = rd.get("metrics/recall(B)", 0)
        metrics["mAP50"] = rd.get("metrics/mAP50(B)", 0)
        metrics["mAP50-95"] = rd.get("metrics/mAP50-95(B)", 0)
    
    # Per-class metrics if available
    per_class = {}
    if hasattr(results, 'box'):
        box = results.box
        if hasattr(box, 'ap50') and box.ap50 is not None:
            class_names = {0: "Pothole", 1: "Crack", 2: "Manhole"}
            for i, name in class_names.items():
                if i < len(box.ap50):
                    per_class[name] = {
                        "AP50": float(box.ap50[i]),
                        "AP50-95": float(box.ap[i]) if hasattr(box, 'ap') and i < len(box.ap) else 0,
                        "precision": float(box.p[i]) if hasattr(box, 'p') and i < len(box.p) else 0,
                        "recall": float(box.r[i]) if hasattr(box, 'r') and i < len(box.r) else 0,
                    }
    
    result_data = {
        "model_name": model_name,
        "model_path": model_path,
        "n_layers": int(n_layers),
        "n_params": int(n_params),
        "n_grads": int(n_grads),
        "gflops": float(gflops),
        "model_size_mb": float(model_size_mb),
        "precision": float(metrics.get("precision", 0)),
        "recall": float(metrics.get("recall", 0)),
        "mAP50": float(metrics.get("mAP50", 0)),
        "mAP50-95": float(metrics.get("mAP50-95", 0)),
        "latency_ms": float(mean_lat),
        "latency_std_ms": float(std_lat),
        "fps": float(fps),
        "per_class": per_class,
    }
    
    output_path = os.path.join(results_dir, f"{model_name}_metrics.json")
    with open(output_path, "w") as f:
        json.dump(result_data, f, indent=2)
    print(f"Results saved to {output_path}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return result_data


## Load and Verify YOLO11s-P2 Model

In [5]:
from ultralytics import YOLO

MODEL_CFG = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\configs\yolo11s-p2.yaml"
MODEL_NAME = "baseline"

# Load model from config
model = YOLO(MODEL_CFG)

# Print model summary
info = model.info(verbose=True)
print(f"\nLayers: {info[0]}, Parameters: {info[1]:,}, Gradients: {info[2]:,}, GFLOPs: {info[3]:.1f}")

# Verify P2 detection scale exists
detect_layer = model.model.model[-1]  # Last layer should be Detect
print(f"\nDetect layer: {detect_layer.__class__.__name__}")
print(f"Number of detection scales: {detect_layer.nl}")
print(f"Detection strides: {detect_layer.stride.tolist() if hasattr(detect_layer, 'stride') else 'N/A'}")

# P2 should give stride=4 (P2/4), P3=8, P4=16, P5=32
assert detect_layer.nl == 4, f"Expected 4 detection scales (P2,P3,P4,P5), got {detect_layer.nl}"
print("\n[OK] P2 detection scale confirmed (4 detection heads)")

# Test forward pass
x = torch.randn(1, 3, 640, 640)
if DEVICE != "cpu":
    x = x.to(f"cuda:{DEVICE}")
    model.model.to(f"cuda:{DEVICE}")

output = model.model(x)
print(f"\nForward pass OK - output type: {type(output)}")


YOLO11s-p2 summary: 217 layers, 9,625,968 parameters, 9,625,952 gradients, 29.7 GFLOPs



Layers: 217, Parameters: 9,625,968, Gradients: 9,625,952, GFLOPs: 29.7

Detect layer: Detect
Number of detection scales: 4
Detection strides: [4.0, 8.0, 16.0, 32.0]

[OK] P2 detection scale confirmed (4 detection heads)



Forward pass OK - output type: <class 'dict'>


## Train Baseline Model

In [6]:
# Train
project_dir = os.path.join(r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning", "runs", "baseline")
train_config = TRAIN_CONFIG.copy()
train_config["project"] = project_dir
train_config["name"] = "train"
train_config["exist_ok"] = True

print(f"Starting BASELINE model training...")

import os
best_path_check = os.path.join(project_dir, 'train', 'weights', 'best.pt')
if os.path.exists(best_path_check):
    print(f"Found {best_path_check}, skipping training!")
    results = None
else:
    results = model.train(**train_config)

print("\nTraining complete!")


Starting baseline training...
Found D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\baseline\train\weights\best.pt, skipping training!

Training complete!


## Evaluate Best Checkpoint

In [7]:
# Load best checkpoint and evaluate
best_path = os.path.join(project_dir, "train", "weights", "best.pt")
if not os.path.exists(best_path):
    # Try alternative path
    import glob
    best_candidates = glob.glob(os.path.join(project_dir, "**/best.pt"), recursive=True)
    if best_candidates:
        best_path = best_candidates[0]
    else:
        raise FileNotFoundError("best.pt not found!")

print(f"Best checkpoint: {best_path}")
print(f"Model size: {os.path.getsize(best_path) / 1024 / 1024:.2f} MB")

# Evaluate on validation set
model = YOLO(best_path)
val_results = model.val(data=TRAIN_CONFIG["data"], imgsz=TRAIN_CONFIG["imgsz"], device=DEVICE, workers=0)

print("\nValidation Results:")
print(f"  Precision: {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall: {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP50: {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP50-95: {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")


Best checkpoint: D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\baseline\train\weights\best.pt
Model size: 18.68 MB


Ultralytics 8.4.92  Python-3.13.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)


YOLO11s-p2 summary (fused): 121 layers, 9,559,900 parameters, 0 gradients, 28.6 GFLOPs


val: Fast image access  (ping: 0.10.0 ms, read: 469.0100.6 MB/s, size: 102.6 KB)


val: Scanning D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\labels\val.cache... 401 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 401/401 51.0Mit/s 0.0s

...

                 Crack        284        527      0.342      0.237      0.195     0.0682


               Manhole        146        193      0.592      0.637      0.632      0.304


Speed: 0.3ms preprocess, 12.1ms inference, 0.0ms loss, 2.4ms postprocess per image


Results saved to D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\detect\val



Validation Results:
  Precision: 0.4491
  Recall: 0.4095
  mAP50: 0.3923
  mAP50-95: 0.1693


## Benchmark Latency

In [8]:
# Run controlled benchmark
benchmark_results = benchmark_model(best_path, DEVICE)


Warming up (20 iterations)...
WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


...

WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.572713851928711. Dividing input by 255.


Latency: 34.43 +/- 6.36 ms
FPS: 29.0


## Save Results

In [9]:
# Save all results
result_data = save_experiment_results(MODEL_NAME, val_results, best_path, benchmark_results, RESULTS_DIR)

# Print summary
print("\n" + "=" * 60)
print("BASELINE RESULTS SUMMARY")
print("=" * 60)
for k, v in result_data.items():
    if k not in ["model_path", "per_class"]:
        print(f"  {k}: {v}")
if result_data.get("per_class"):
    print("\nPer-class results:")
    for cls_name, cls_metrics in result_data["per_class"].items():
        print(f"  {cls_name}: AP50={cls_metrics['AP50']:.4f}, AP50-95={cls_metrics['AP50-95']:.4f}")

# Cleanup
del model, val_results
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nDone!")


YOLO11s-p2 summary: 217 layers, 9,575,292 parameters, 0 gradients, 29.0 GFLOPs


Results saved to D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results\baseline_metrics.json



BASELINE RESULTS SUMMARY
  model_name: baseline
  n_layers: 217
  n_params: 9575292
  n_grads: 0
  gflops: 28.953856000000002
  model_size_mb: 18.680789947509766
  precision: 0.44906048649047864
  recall: 0.40950398295863377
  mAP50: 0.39231973458348873
  mAP50-95: 0.16934975216430745
  latency_ms: 34.427608999976655
  latency_std_ms: 6.355074652679429
  fps: 29.046455128518453

Per-class results:
  Pothole: AP50=0.3499, AP50-95=0.1356
  Crack: AP50=0.1949, AP50-95=0.0682
  Manhole: AP50=0.6321, AP50-95=0.3043



Done!
